# Random Forest implementation and tuning

Fit and evaluate a parcel-routing classifier end to end. Start with a single-tree baseline, fit a forest, tune using randomized cross-validation, then assess the selected model on held-out data. Finish by inspecting an out-of-bag prediction and two types of feature importance.

All observations are original synthetic examples generated below. The simulation contains overlapping classes, nonlinear structure, and an irrelevant numeric column. The displayed metrics describe this dataset and split. Run all cells in order from a clean runtime. No external data files are needed.

In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier

print(f'NumPy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}')

## Generate a training problem and hold out test parcels

`review` is the recorded route, expressed as 1 for **Review** and 0 for **Auto**. Signals include parcel weight, scanner score, seal quality and handling details. Low seal quality and certain combinations of weight and scan score increase the simulated chance of Review. `noise_measure` is unrelated to the simulated route.

Make a stratified split before training so the class balance stays approximately consistent. Keep the test rows separate during model selection. Use balanced accuracy as the selection target so the two classes count equally, then report Review recall to make missed reviews visible.

In [ ]:
seed = 37
rng = np.random.default_rng(seed)
n_rows = 1400
X = pd.DataFrame({
    'weight': rng.uniform(0.5, 10, n_rows),
    'scan_score': rng.uniform(0, 10, n_rows),
    'seal_quality': rng.uniform(0, 10, n_rows),
    'handling_steps': rng.integers(1, 8, n_rows),
    'fragility_score': rng.uniform(0, 10, n_rows),
    'noise_measure': rng.random(n_rows),
})
heavy_scan = (X['weight'] > 5.7) & (X['scan_score'] > 5.8)
fragile_handling = (X['fragility_score'] > 6) & (X['handling_steps'] > 4)
log_odds = (-2.4 + 3.8 * heavy_scan + 2.6 * (X['seal_quality'] < 3.2)
            + 2.0 * fragile_handling + 0.08 * (X['scan_score'] - 5))
risk = 1 / (1 + np.exp(-log_odds))
y = pd.Series(rng.binomial(1, risk), name='review', index=X.index)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=seed, stratify=y,
)
display(X_train.assign(Review=y_train).head().round(2))
print(f'{len(X_train)} training rows | {len(X_test)} held-out rows')
print(f'Observed Review share in training: {y_train.mean():.1%}')

## Fit the two baselines

The first baseline is one tree. The forest uses bootstrap rows and considers a subset of candidate features at each node. The same training data will serve both baselines and all cross-validation trials.

In [ ]:
tree_baseline = DecisionTreeClassifier(random_state=seed).fit(X_train, y_train)
forest_baseline = RandomForestClassifier(
    n_estimators=100, bootstrap=True, max_features='sqrt', random_state=seed, n_jobs=1,
).fit(X_train, y_train)
print(f'Baseline tree depth: {tree_baseline.get_depth()}')
print(f'Baseline forest trees: {len(forest_baseline.estimators_)}')

## Tune using randomized search on the training partition

A hyperparameter is a setting chosen before fitting. The parameter space below provides plausible settings. Randomized search draws a fixed budget of combinations, tests each by stratified four-fold cross-validation, and refits the best-scoring candidate on all training rows. With lists for all parameters, the search samples combinations without replacement. The hold-out set remains separate through this step.

**The five knobs in this lesson**

- `n_estimators` controls the number of trees. More trees stabilize the average of the forest's randomized predictions and cost extra fit time, prediction time, and memory. Improvements commonly level off.
- `max_samples` controls bootstrap draws per tree when bootstrapping is enabled. Fewer draws generally make bags less alike and give each tree less data. Unique row counts depend on repeat draws.
- `max_features` controls the candidate feature budget at each split. A smaller budget can reduce similarity between trees and can make individual questions weaker. The splitter can inspect further features when it needs a valid split.
- `max_depth` caps the depth during growth. A shallower tree often gives simpler, less variable predictions, with a greater risk of missing structure.
- `ccp_alpha` weights the number of leaves in cost-complexity pruning. Larger values favor smaller trees. Choose a value by validation so the forest can still capture useful structure.

These knobs interact. Training and validation scores help reveal whether a setting changes fit, and repeated training runs let you inspect the spread of predictions directly.

In [ ]:
parameter_space = {
    'n_estimators': [64, 100, 140],
    'max_samples': [0.6, 0.8, None],
    'max_features': [1, 'sqrt', None],
    'max_depth': [4, 7, 11, None],
    'ccp_alpha': [0.0, 0.0005, 0.002],
}
folds = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
search = RandomizedSearchCV(
    RandomForestClassifier(bootstrap=True, oob_score=True, random_state=seed, n_jobs=1),
    parameter_space, n_iter=15, scoring='balanced_accuracy', cv=folds,
    refit=True, return_train_score=True, random_state=seed, n_jobs=1, error_score='raise',
).fit(X_train, y_train)

forest = search.best_estimator_
results = pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
columns = ['rank_test_score', 'mean_train_score', 'mean_test_score', 'std_test_score']
display(results[columns + [f'param_{key}' for key in parameter_space]].head().round(4))
print('Selected settings:', search.best_params_)
print(f'Selected mean CV balanced accuracy: {search.best_score_:.4f}')

## Evaluate the selected forest on the hold-out set

Accuracy counts correct predictions overall. Balanced accuracy averages the recall for Auto and the recall for Review. Review recall is the share of actual Review parcels caught by the model. In the confusion matrix, an actual Review predicted Auto is a missed review. An actual Auto predicted Review is an unnecessary manual review.

Cross-validation selected the settings. The metrics below evaluate all three fitted models on the same 350 test parcels. A tuned model can achieve a lower test score than a baseline because test performance varies with the particular held-out sample.

In [ ]:
fitted = {'Single tree': tree_baseline, 'Baseline forest': forest_baseline, 'Tuned forest': forest}
test_predictions = {name: model.predict(X_test) for name, model in fitted.items()}
scores = pd.DataFrame.from_dict({name: {
    'accuracy': accuracy_score(y_test, pred),
    'balanced accuracy': balanced_accuracy_score(y_test, pred),
    'Review recall': recall_score(y_test, pred, pos_label=1),
} for name, pred in test_predictions.items()}, orient='index')
display(scores.round(4))

matrix = confusion_matrix(y_test, test_predictions['Tuned forest'], labels=[0, 1])
display(pd.DataFrame(matrix, index=['Actual Auto', 'Actual Review'],
                     columns=['Predicted Auto', 'Predicted Review']))

## Read the winning class from the mean probabilities

The forest class order is `forest.classes_`. Each tree returns scores in the encoded order used during the forest fit. For this 0/1 target those columns match the forest's Auto and Review columns. Average the probability vectors, then choose the corresponding label from `classes_`. The assertion checks that this manual result matches `predict_proba`.

In [ ]:
example = X_test.iloc[[0]]
mean_prob = np.mean([est.predict_proba(example.to_numpy())[0] for est in forest.estimators_], axis=0)
np.testing.assert_allclose(mean_prob, forest.predict_proba(example)[0])
np.testing.assert_array_equal(forest.classes_[np.argmax(mean_prob)], forest.predict(example)[0])
display(example.round(2))
display(pd.DataFrame({'class label': forest.classes_, 'mean probability': mean_prob}).round(4))

## Out-of-bag scoring from a row's eligible trees

Each tree trains on a bootstrap sample. A row is out of bag for a tree when its position appears zero times in that tree's sample. To predict a training row out of bag, average predictions from only the trees that omitted it. Different rows use different groups of trees. With a small forest, some rows may have no eligible trees, so check coverage before interpreting the result.

For uniform sampling, a particular row has probability $(1-1/n)^m$ of being omitted in $m$ draws from $n$ rows. For $m=n$ and large $n$, this is about 36.8%. The selected model may use a smaller sample size. The held-out result above uses a separate partition and every tree can predict each held-out row.

In [ ]:
bags = forest.estimators_samples_
in_bag_counts = np.stack([np.bincount(bag, minlength=len(X_train)) for bag in bags])
eligible_counts = (in_bag_counts == 0).sum(axis=0)
eligible_rows = eligible_counts > 0
position = int(np.flatnonzero(eligible_rows)[0])
eligible_trees = np.flatnonzero(in_bag_counts[:, position] == 0)
row = X_train.iloc[[position]].to_numpy()
manual_oob = np.mean([forest.estimators_[i].predict_proba(row)[0] for i in eligible_trees], axis=0)
np.testing.assert_allclose(manual_oob, forest.oob_decision_function_[position])

oob_probs = forest.oob_decision_function_[eligible_rows]
oob_predictions = forest.classes_[oob_probs.argmax(axis=1)]
covered_accuracy = accuracy_score(y_train.to_numpy()[eligible_rows], oob_predictions)
print(f'Training parcel at row position {position} is omitted by {len(eligible_trees)} trees.')
print('First eligible tree numbers (1-based):', (eligible_trees[:6] + 1).tolist())
print('That parcel’s OOB class probabilities:', np.round(manual_oob, 4))
print(f'OOB coverage: {int(eligible_rows.sum())} of {len(eligible_rows)} training rows')
print(f'OOB accuracy on covered rows: {covered_accuracy:.4f}')
print(f'Built-in OOB accuracy: {forest.oob_score_:.4f}')
if eligible_rows.all():
    np.testing.assert_allclose(covered_accuracy, forest.oob_score_)

## Explain forest feature importance

Impurity-based importance averages the normalized credits for training impurity reduction across trees that split, then normalizes the average. A split receives credit based on the impurity reduction and the fraction of training samples that reach it. Features with many available split points can receive inflated impurity-based scores.

Permutation importance measures the drop in a chosen score after shuffling one feature on held-out rows. Here the score is balanced accuracy and we repeat each shuffle 10 times. Redundant or correlated features can share predictive information and look weaker one at a time. Both measures describe this fitted model and evaluation setup.

In [ ]:
permutation = permutation_importance(
    forest, X_test, y_test, scoring='balanced_accuracy', n_repeats=10, random_state=seed, n_jobs=1,
)
importance = pd.DataFrame({
    'impurity importance': forest.feature_importances_,
    'mean held-out score drop': permutation.importances_mean,
    'score-drop spread': permutation.importances_std,
}, index=X.columns).sort_values('mean held-out score drop', ascending=False)
display(importance.round(4))

## Explain the pipeline back in one minute

Give each tree a bootstrap sample and a random candidate feature subset at each split, then average the trees' probabilities. Tune settings by cross-validation on the training partition. Assess the selected model on held-out parcels. Use OOB and importance scores to answer complementary questions about coverage and model behavior.

**Reference run.** Python 3.12 and scikit-learn 1.6.1 produced 0.8126 mean cross-validation balanced accuracy for the selected model. Held-out balanced accuracy was 0.7265 for the single tree, 0.7689 for the baseline forest, and 0.7888 for the tuned forest. The tuned forest's OOB accuracy was 0.8124 with all 1,050 training rows covered. These are checkpoints for the seeded synthetic example. Other library versions can give slightly different model results.

[Random Forest API](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) · [RandomizedSearchCV API](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) · [Permutation importance guide](https://scikit-learn.org/stable/modules/permutation_importance.html)